# 선정 쪽수 전체 청크 → 적재 패키지

manifest `선정쪽수`의 모든 쪽을 청크로 만들고(표 행 = 표 작업본, 문단 = 본문 변환본), 이미 승인·정규화한 학교 행동 42개와 합쳐 **하나의 적재 패키지**를 만든다. 적재는 `school_load.ipynb`에서 이 패키지로 한다.

- 규칙은 `src/preprocess/school/chunks.py` 머리말 참고. 쪽 번호는 사용자가 PDF와 대조 완료(11/4).
- 본문 변환본 속 뒤섞인 표는 표 작업본 내용과 겹치는 줄을 빼서 두 번 적재하지 않는다.
- 질병 연결(ABOUT)은 표 행의 첫 칸·표 제목이 질병명일 때만.
- 표 행의 section은 '본문 장·절 경로 > 표 제목'(표 제목에 단계가 없어도 원문 단계를 알 수 있게).
- 검사: 골든 Citation 42건이 모두 해당 청크 원문에 있는지, 기존 청크 4개가 같은 ID·본문으로 재현되는지, ID 장부와 충돌이 없는지.

In [1]:
import json
import re
import sys
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from index.school.bundle import build_load_bundle
from index.school.dictionaries import load_dictionaries
from index.school.id_map import IdRegistry, document_node, load_manifest
from index.school.normalize import read_jsonl, write_jsonl
from preprocess.school.chunks import build_all_chunks

NORMALIZED = ROOT / "data" / "interim" / "school" / "normalized_20261104T063559_886830Z"   # 승인 행동 42개
books = load_dictionaries()
manifest = load_manifest(ROOT)
chunks, about = build_all_chunks(ROOT, manifest, books)
print("청크:", len(chunks), dict(Counter((c["document_id"], c["kind"]) for c in chunks)))
print("ABOUT:", len(about), dict(Counter(v[0] for v in about.values())))
lengths = [len(c["text"]) for c in chunks if c["kind"] == "paragraph"]
print("문단 길이 최소/중앙/최대:", min(lengths), sorted(lengths)[len(lengths) // 2], max(lengths))

청크: 182 {('moe_school_manual_v3', 'table_row'): 52, ('moe_school_manual_v3', 'paragraph'): 28, ('gyeonggi_school_guide_2025', 'table_row'): 80, ('gyeonggi_school_guide_2025', 'paragraph'): 22}
ABOUT: 31 {'TUBERCULOSIS': 12, 'VARICELLA': 1, 'EKC': 1, 'MUMPS': 1, 'INFLUENZA': 1, 'SCARLET_FEVER': 14, 'HFMD': 1}
문단 길이 최소/중앙/최대: 35 242 637


## 검사 1 - 골든 Citation 42건과 기존 청크

In [2]:
squash = lambda s: re.sub(r"\s+", "", s.replace("<br>", ""))
golden = (ROOT / "data" / "specs" / "golden" / "school.md").read_text(encoding="utf-8")
rows = re.findall(r"^\| (C\d\d) \|[^|]*\| ([a-z0-9_]+) \| ([a-z0-9_]+:p\d{4}:[a-z0-9]+) \| (\d+) \| [^|]* \| [^|]* \| (.+?) \| ",
                  golden, re.M)
by_id = {c["chunk_id"]: c for c in chunks}
check = pd.DataFrame([{"CID": cid, "chunk_id": chunk, "청크 있음": chunk in by_id,
                       "excerpt 포함": chunk in by_id and squash(excerpt) in squash(by_id[chunk]["text"])}
                      for cid, doc, chunk, page, excerpt in rows])
print(f"골든 Citation {len(check)}건 / 청크 있음 {int(check['청크 있음'].sum())} / excerpt 포함 {int(check['excerpt 포함'].sum())}")
assert len(check) == 42 and check["excerpt 포함"].all()

old = {c["chunk_id"]: c for c in read_jsonl(NORMALIZED / "official_chunks.jsonl")}
same = {cid: by_id[cid]["text"] == c["text"] for cid, c in old.items()}
print("기존 청크 본문 재현:", same)
print("section 변경(표 행에 본문 절 경로를 붙임):", {cid: (c["section"], by_id[cid]["section"]) for cid, c in old.items()
                                               if by_id[cid]["section"] != c["section"]})
assert all(same.values())

골든 Citation 42건 / 청크 있음 42 / excerpt 포함 42
기존 청크 본문 재현: {'moe_school_manual_v3:p0036:t01r01': True, 'moe_school_manual_v3:p0036:t01r02': True, 'gyeonggi_school_guide_2025:p0054:t01r01': True, 'gyeonggi_school_guide_2025:p0054:t01r02': True}
section 변경(표 행에 본문 절 경로를 붙임): {'moe_school_manual_v3:p0036:t01r01': ('모의 훈련 시나리오: 대응 1단계 주요 조치사항', '가. 대응 1단계: 학교 내 유증상자 발견 시나리오 > 모의 훈련 시나리오: 대응 1단계 주요 조치사항'), 'moe_school_manual_v3:p0036:t01r02': ('모의 훈련 시나리오: 대응 1단계 주요 조치사항', '가. 대응 1단계: 학교 내 유증상자 발견 시나리오 > 모의 훈련 시나리오: 대응 1단계 주요 조치사항')}


## 검사 2 - ID 장부(같은 ID인데 원문이 바뀌면 오류)

In [3]:
registry = IdRegistry.load()
before = len(registry.data["chunks"])
for chunk in chunks:
    registry.register_chunk(chunk)
print(f"장부 청크 {before} → {len(registry.data['chunks'])}")
registry.save()

장부 청크 182 → 182


WindowsPath('C:/mle-02-p2-team2/src/index/school/id_registry.json')

## 적재 패키지

In [4]:
actions = read_jsonl(NORMALIZED / "normalized.jsonl")
documents = [document_node(d, manifest) for d in sorted({c["document_id"] for c in chunks})]
bundle = build_load_bundle(actions, chunks, documents, books, concept_policy="omit_link", chunk_diseases=about)
print("노드:", {k: len(v) for k, v in bundle["nodes"].items()})
print("관계:", dict(Counter(e["type"] for e in bundle["edges"])))
print("DB에 먼저 있어야 할 사전 노드:", len(bundle["dictionary_refs"]))
out_dir = ROOT / "data" / "interim" / "school" / datetime.now(UTC).strftime("full_%Y%m%dT%H%M%S_%fZ")
out_dir.mkdir(parents=True)
write_jsonl(out_dir / "official_chunks.jsonl", chunks)
(out_dir / "bundle.json").write_text(json.dumps(bundle, ensure_ascii=False, indent=1, allow_nan=False), encoding="utf-8")
(out_dir / "meta.json").write_text(json.dumps({"actions_from": NORMALIZED.name, "created_at": datetime.now(UTC).isoformat(),
    "chunks": len(chunks), "about": len(about)}, ensure_ascii=False, indent=1), encoding="utf-8")
print("저장:", out_dir.relative_to(ROOT))

노드: {'Document': 2, 'Chunk': 182, 'SchoolAction': 42}
관계: {'HAS_CHUNK': 182, 'ABOUT': 31, 'SUPPORTED_BY': 42, 'DONE_BY': 42, 'INSTANCE_OF': 19}
DB에 먼저 있어야 할 사전 노드: 16
저장: data\interim\school\full_20261104T075351_804218Z
